In [1]:
# %% [markdown]
# # 03c — Sparse KAN v2 (Retrain-Only, Fixed Init + Corrected eps)
#
# ═══════════════════════════════════════════════════════════════════════
# WHAT THIS NOTEBOOK IS AND WHY IT EXISTS
# ═══════════════════════════════════════════════════════════════════════
#
# TWO CONFIRMED BUGS, fixed in sparse_kan.py since the original sweep:
#
# BUG 1 — Masked-weight reinitialisation was a silent no-op.
#   `self.base_weight.data[q, active].uniform_(-bound, bound)` used a
#   boolean mask for advanced indexing, which returns a COPY -- the
#   in-place .uniform_() wrote to that discarded copy and never touched
#   the real tensor. Every active edge silently kept efficient-kan's own
#   default init (scaled by full in_features=1699/574), not the correct
#   per-node fan-in scale. Measured directly: mean ratio of
#   actual-max-weight to theoretical-fan-in-bound was 0.058 (should be
#   ~1.0) across 145 real subthemes with fan_in >= 5, confirmed on both
#   base_weight and spline_scaler. FIXED by building a fresh row tensor
#   and assigning it back via __setitem__, which writes through correctly.
#
# BUG 2 — BatchNorm's eps=0.1 was dominating the normalisation for any
#   unit with true variance below ~0.1 (which, given Bug 1, was most of
#   them). y = (x-mu)/sqrt(var+eps): when var << eps, the denominator is
#   set almost entirely by eps, not by the unit's actual data -- so
#   post-BN scale was compressed toward sqrt(eps) regardless of signal,
#   for the entire training run, on every batch. Confirmed via direct
#   forward-pass measurement matching the sigma/sqrt(eps) prediction to
#   3 decimal places.
#
# FIX FOR BUG 2: eps swept empirically from 1e-1 to 1e-7, on real
# per-unit activations, both BatchNorm layers (bn1: subtheme scores,
# bn2: theme scores), both datasets, 3 seeds. Chosen value: eps=1e-5.
# Rationale: clamp rate was found to be governed by genuine tail shape
# in specific units, not by eps (moved <0.1 percentage points across
# 5 orders of magnitude, fully saturated by 1e-5) -- ruling out the
# concern that a small eps causes blow-ups. At eps=1e-5, median unit-std
# was exactly 1.000 on both layers/datasets; the small number of units
# still below 0.9 are near-degenerate subthemes with near-total sign
# cancellation among highly correlated (rho>0.9) active inputs -- an
# initialisation artefact of redundant feature groups (confirmed to vary
# by random seed, not a fixed taxonomy defect), not something a smaller
# eps should be chosen to rescue.
#
# ═══════════════════════════════════════════════════════════════════════
# WHY RETRAIN-ONLY, NOT A FRESH OPTUNA SEARCH
# ═══════════════════════════════════════════════════════════════════════
#
# This notebook reuses the best hyperparameters already stored in the
# ORIGINAL Sparse KAN run's Optuna .db files (both no-L1 and with-L1
# phases, per config, per seed) rather than re-searching. This is a
# deliberate methodological choice, not a shortcut:
#
#   1. Speed: Optuna search was ~98% of the original sweep's runtime;
#      retrain-only for all 48 configs x 3 seeds takes roughly 40 min.
#   2. Clean attribution: any change in results between the original
#      (buggy-init) checkpoints and these (fixed-init) ones can only be
#      attributed to the init/eps fix, not to additional tuning budget.
#      This is stated explicitly in the methods section: "hyperparameters
#      were selected under the original (pre-fix) configuration and held
#      fixed for the corrected retrain; no additional search budget was
#      spent."
#
# ═══════════════════════════════════════════════════════════════════════
# RESULTS ISOLATION -- CRITICAL
# ═══════════════════════════════════════════════════════════════════════
#
# This notebook writes to a COMPLETELY SEPARATE results directory
# (.../sparse_kan_v2/) from the original (.../sparse_kan/). The original
# checkpoints, predictions, and Optuna .db files are NEVER touched,
# NEVER overwritten -- they are kept deliberately as the "before" side
# of a controlled ablation (broken-init vs fixed-init, identical
# hyperparameters). This notebook only ever READS from the original
# results directory's optuna/ subfolder, and only ever WRITES to the
# new v2 directory.
#
# Estimated runtime: ~40 minutes for all 48 configs x 3 seeds
# (retrain only, no search).

# %%
# ── COLAB SETUP ──
!pip install -q git+https://github.com/Blealtan/efficient-kan.git optuna

from google.colab import drive
drive.mount("/content/drive")

import sys
sys.path.insert(0, "/content/drive/MyDrive/Thesis/Code")

# %%
import json
import numpy as np
import pandas as pd
import random
import time
import torch
import optuna
from pathlib import Path

from data_utils import load_split, get_dataloaders, get_device, load_theme_assignment
from training import train_model, save_checkpoint
from evaluation import (
    evaluate_model, save_predictions, compute_calibration,
    load_predictions, run_full_backtest,
)
from sparse_kan import SparseKAN, sparse_kan_edge_l1

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ═══════════════════════════════════════════════════════════════════════════
# SANITY CHECK — confirm the live sparse_kan.py actually has BOTH fixes,
# before spending any GPU time retraining against a stale file.
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 90)
print("SANITY CHECK: confirming init fix + eps=1e-5 are both present")
print("=" * 90)

# ── Fixture note: in_features MUST be >> fan_in for this check to work.
# The bug substituted 1/sqrt(in_features) for 1/sqrt(fan_in), so a small
# fixture (in_features ~ fan_in) makes broken and fixed indistinguishable
# and the assert passes on stale code. 200 features vs fan_in=5 gives a
# ~6x separation: fixed -> ratio ~0.83, broken -> ratio ~0.14.
N_DUMMY = 200
_cols = [f"f{i}" for i in range(N_DUMMY)]
_fake_tax = pd.DataFrame({
    "column":        _cols,
    "subtheme_id":   ["01_01"]*5 + [f"02_{i:02d}" for i in range(N_DUMMY - 5)],
    "subtheme_name": ["SubA"]*5  + ["SubB"]*(N_DUMMY - 5),
    "theme_id":      ["01"]*5    + ["02"]*(N_DUMMY - 5),
    "theme_name":    ["ThemeX"]*5 + ["ThemeY"]*(N_DUMMY - 5),
})

_m = SparseKAN.from_taxonomy(_fake_tax, _cols, grid_size=5, spline_order=3,
                             grid_range=[-1, 1])

# Check 1: eps is 1e-5, not the old 0.1
assert abs(_m.bn1.eps - 1e-5) < 1e-9, (
    f"bn1.eps={_m.bn1.eps}, expected 1e-5 -- sparse_kan.py on Drive is STALE. STOP."
)
assert abs(_m.bn2.eps - 1e-5) < 1e-9, f"bn2.eps={_m.bn2.eps}, expected 1e-5"
print(f"  ✓ bn1.eps = {_m.bn1.eps}, bn2.eps = {_m.bn2.eps}")

# Check 1b: affine=False not silently reverted to the default True
assert _m.bn1.affine is False, f"bn1.affine={_m.bn1.affine}, expected False. STOP."
assert _m.bn2.affine is False, f"bn2.affine={_m.bn2.affine}, expected False. STOP."
print(f"  ✓ bn1.affine = {_m.bn1.affine}, bn2.affine = {_m.bn2.affine}")

# Check 2: init fix present on BOTH branches. spline_scaler gates the
# spline contribution -- if only base_weight were fixed, the splines
# would still be suppressed and the retrain would miss its target.
_active = _m.layer0.mask[0].bool()
_fan_in = int(_active.sum().item())
assert _fan_in == 5, f"fixture built wrong: fan_in={_fan_in}, expected 5"
_bound = 1.0 / _fan_in**0.5
# E[max of n uniform draws] = bound * n/(n+1) = 0.833 * bound when fixed.
# Broken: 1/sqrt(200) / (1/sqrt(5)) * 5/6 = 0.132.
for _pname, _W in [("base_weight",   _m.layer0.base_weight.data),
                   ("spline_scaler", _m.layer0.spline_scaler.data)]:
    _ratio = _W[0][_active].abs().max().item() / _bound
    assert _ratio > 0.5, (
        f"{_pname} ratio={_ratio:.3f} -- expected ~0.83 (fixed) vs ~0.13 "
        f"(broken). sparse_kan.py on Drive is STALE. STOP. Re-upload."
    )
    print(f"  ✓ init fix on {_pname}: ratio = {_ratio:.3f} "
          f"(fan_in={_fan_in}, expect ~0.83)")

# Check 3: masked entries zero and finite, and a train-mode forward pass
# works (BatchNorm needs batch > 1)
assert _m.verify_masking(), "masking broken at construction. STOP."
_m.train()
assert _m(torch.randn(16, N_DUMMY)).shape == (16, 1)
print("  ✓ forward pass OK in train mode (batch=16)")

del _fake_tax, _cols, _m, _active, _fan_in, _bound, N_DUMMY

print("\n" + "=" * 90)
print("SANITY CHECK PASSED -- safe to proceed with retrain-only sweep")
print("=" * 90)


# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

SPLITS_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")
THEMES_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
HUBER_DELTA_PATH = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/03_targets/huber_delta.json")

# ── READ from the ORIGINAL results (Optuna .db files only) ──
ORIGINAL_RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan")

# ── WRITE to a completely separate v2 directory. NEVER touches the original. ──
RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan_v2")

DATASETS     = ["agg_full_moments", "agg_means"]
TARGET_TYPES = ["binary", "continuous"]
ALL_SPLITS   = ["Split_A", "Split_B", "Split_C", "Split_D"]

SEEDS = [42, 123, 456]

GRID_SIZE    = 14
SPLINE_ORDER = 3
GRID_RANGE   = [-5.5, 5.5]

ACTIVATION_PROBE_N = 2048

with open(HUBER_DELTA_PATH) as f:
    HUBER_DELTAS = json.load(f)["deltas"]

def get_huber_delta(split_name):
    return HUBER_DELTAS[f"{split_name}/market"]


# ═══════════════════════════════════════════════════════════════════════════
# REPRODUCIBILITY
# ═══════════════════════════════════════════════════════════════════════════

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ═══════════════════════════════════════════════════════════════════════════
# LOAD TAXONOMIES
# ═══════════════════════════════════════════════════════════════════════════

print("Loading taxonomies...")
taxonomy_dfs = {}
for ds in DATASETS:
    df = load_theme_assignment(ds, THEMES_DIR)
    taxonomy_dfs[ds] = df
    print(f"  {ds}: {len(df)} features, "
          f"{df['subtheme_id'].nunique()} subthemes, "
          f"{df['theme_id'].nunique()} themes")

device = get_device()


# ═══════════════════════════════════════════════════════════════════════════
# ACTIVATION MONITORING (same as before, unchanged)
# ═══════════════════════════════════════════════════════════════════════════

def make_activation_callback(probe_batch, device):
    @torch.no_grad()
    def callback(model, epoch):
        model.eval()
        x0 = probe_batch.to(device)
        x1 = model.layer0(x0)
        x1n_preclamp = model.bn1(x1)
        x1n = torch.clamp(x1n_preclamp, min=-5.0, max=5.0)
        x2 = model.layer1(x1n)
        x2n_preclamp = model.bn2(x2)
        x2n = torch.clamp(x2n_preclamp, min=-5.0, max=5.0)
        model.train()

        clamp1_rate = (x1n_preclamp.abs() > 5.0).float().mean().item()
        clamp2_rate = (x2n_preclamp.abs() > 5.0).float().mean().item()

        print(f"    [activations @ epoch {epoch}]  "
              f"L1 pre-BN max|x|={x1.abs().max().item():7.2f} std={x1.std().item():6.3f}  "
              f"post-BN(pre-clamp) max|x|={x1n_preclamp.abs().max().item():5.2f} "
              f"std={x1n_preclamp.std().item():5.3f}  clamp_rate={clamp1_rate:.3%}  |  "
              f"L2 pre-BN max|x|={x2.abs().max().item():7.2f} std={x2.std().item():6.3f}  "
              f"post-BN(pre-clamp) max|x|={x2n_preclamp.abs().max().item():5.2f} "
              f"std={x2n_preclamp.std().item():5.3f}  clamp_rate={clamp2_rate:.3%}")
    return callback


# ═══════════════════════════════════════════════════════════════════════════
# RETRAIN-ONLY: load stored best params, skip Optuna search entirely
# ═══════════════════════════════════════════════════════════════════════════

def load_best_params_from_original(model_name, target_type, split_name, seed):
    """
    Reads the ORIGINAL run's Optuna studies (no_L1 and with_L1 phases) for
    this exact config, picks whichever phase originally won, and returns
    its best_params -- WITHOUT running any new trials. Raises loudly if
    the original study is missing, rather than silently falling back to a
    fresh search (which would break the "no additional tuning" guarantee).
    """
    study_dir = ORIGINAL_RESULTS_DIR / f"seed_{seed}" / "optuna"

    path_no_l1 = study_dir / f"{model_name}_{target_type}_{split_name}_no_L1_seed{seed}.db"
    path_l1    = study_dir / f"{model_name}_{target_type}_{split_name}_with_L1_seed{seed}.db"

    if not path_no_l1.exists() or not path_l1.exists():
        raise FileNotFoundError(
            f"Original Optuna studies not found for {model_name}/{target_type}/"
            f"{split_name}/seed{seed} at {study_dir} -- cannot retrain-only "
            f"without the original hyperparameter search results."
        )

    study_no_l1 = optuna.load_study(
        study_name=f"{model_name}_{target_type}_{split_name}_no_L1_seed{seed}",
        storage=f"sqlite:///{path_no_l1}",
    )
    study_l1 = optuna.load_study(
        study_name=f"{model_name}_{target_type}_{split_name}_with_L1_seed{seed}",
        storage=f"sqlite:///{path_l1}",
    )

    if study_no_l1.best_value >= study_l1.best_value:
        return study_no_l1.best_params, False, study_no_l1.best_value
    else:
        return study_l1.best_params, True, study_l1.best_value


def run_single_retrain(split_name, dataset, target_type, device,
                       seed, seed_results_dir):
    model_name = f"sparse_kan_{dataset}"

    print(f"\n{'─'*60}")
    print(f"  seed={seed} / {dataset} / {split_name} / {target_type}")
    print(f"{'─'*60}")

    data         = load_split(split_name, dataset, SPLITS_DIR)
    feature_cols = data["feature_cols"]
    taxonomy_df  = taxonomy_dfs[dataset]

    n_pos      = data["y_train"].sum()
    n_neg      = len(data["y_train"]) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)

    huber_delta = get_huber_delta(split_name) if target_type == "continuous" else None

    best_params, use_l1, original_val = load_best_params_from_original(
        model_name, target_type, split_name, seed
    )
    print(f"  Reusing stored params (original val={original_val:.4f}, "
          f"used_l1={use_l1}): {best_params}")

    rng = np.random.default_rng(seed)
    n_avail = data["X_train"].shape[0]
    probe_idx = rng.choice(n_avail, size=min(ACTIVATION_PROBE_N, n_avail), replace=False)
    activation_probe = torch.tensor(data["X_train"][probe_idx], dtype=torch.float32)

    batch_size = best_params.get("batch_size", 128)
    loaders    = get_dataloaders(
        split_name, dataset, SPLITS_DIR,
        target_type=target_type,
        batch_size=batch_size,
    )

    model = SparseKAN.from_taxonomy(
        taxonomy_df, feature_cols,
        grid_size=GRID_SIZE, spline_order=SPLINE_ORDER, grid_range=GRID_RANGE,
    )
    activation_callback = make_activation_callback(activation_probe, device)

    final_train_kwargs = {
        "lr":              best_params["lr"],
        "weight_decay":    best_params["weight_decay"],
        "pos_weight":      pos_weight if target_type == "binary" else None,
        "n_epochs":        300,
        "patience":        20,
        "verbose":         True,
        "log_every":       20,
        "epoch_callback":  activation_callback,
    }
    if use_l1:
        final_train_kwargs["reg_fn"]     = sparse_kan_edge_l1
        final_train_kwargs["reg_weight"] = best_params["reg_weight"]
    if target_type == "continuous":
        final_train_kwargs["huber_delta"] = huber_delta

    result = train_model(
        model=model,
        train_loader=loaders["train"],
        val_loader=loaders["val"],
        device=device,
        target_type=target_type,
        **final_train_kwargs,
    )

    assert model.verify_masking(), (
        "Masked weights non-zero after final training completed -- "
        "structural sparsity has been broken somewhere."
    )

    all_metrics = {}
    for part in ["train", "val", "test"]:
        metrics = evaluate_model(
            model, loaders[part], device, target_type,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )
        if target_type == "binary":
            cal = compute_calibration(metrics["y_true"], metrics["y_prob"])
            metrics["ece"] = cal["ece"]

        save_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type=target_type,
            part=part,
            dates=data[f"dates_{part}"],
            returns=data[f"returns_{part}"],
            metrics=metrics,
            hyperparameters=(
                {**best_params, "used_l1": use_l1, "seed": seed,
                 "huber_delta": huber_delta,
                 "reused_from_original": True,
                 "original_val_metric": original_val}
                if part == "test" else None
            ),
            results_dir=seed_results_dir,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )
        all_metrics[part] = metrics

    ckpt_dir = seed_results_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_checkpoint(
        model=model,
        train_result=result,
        hyperparameters={**best_params, "used_l1": use_l1, "seed": seed,
                         "huber_delta": huber_delta,
                         "reused_from_original": True},
        model_config={
            "type":                "SparseKAN_Masked_BN_v2",
            "dataset":             dataset,
            "target_type":         target_type,
            "n_features":          data["n_features"],
            "n_subthemes":         model.n_subthemes,
            "n_themes":            model.n_themes,
            "grid_size":           GRID_SIZE,
            "spline_order":        SPLINE_ORDER,
            "grid_range":          GRID_RANGE,
            "eps":                 1e-5,
            "active_edges":        model.count_active_edges(),
            "total_edges":         model.count_total_edges(),
            "active_parameters":   model.count_active_parameters(),
        },
        path=ckpt_dir / f"{model_name}_{target_type}_{split_name}.pt",
    )

    if target_type == "binary":
        print(f"\n  Results (seed={seed}):")
        print(f"    Train AUC: {all_metrics['train']['auc']:.4f}")
        print(f"    Val AUC:   {all_metrics['val']['auc']:.4f}")
        print(f"    Test AUC:  {all_metrics['test']['auc']:.4f}")
        print(f"    Gap:       {all_metrics['train']['auc'] - all_metrics['test']['auc']:+.4f}")
    else:
        print(f"\n  Results (seed={seed}, huber_delta={huber_delta:.4f}):")
        print(f"    Train R2: {all_metrics['train']['r2']:.4f}")
        print(f"    Val R2:   {all_metrics['val']['r2']:.4f}")
        print(f"    Test R2:  {all_metrics['test']['r2']:.4f}")
        print(f"    Derived AUC: {all_metrics['test']['derived_auc']:.4f}")

    return {
        "best_params": best_params,
        "used_l1":     use_l1,
        "metrics":     all_metrics,
        "best_epoch":  result["best_epoch"],
        "total_time":  result["total_time"],
    }


# %% [markdown]
# ## Run Retrain-Only Sweep (48 configs x 3 seeds, no Optuna search)

# %%
configs    = len(DATASETS) * len(TARGET_TYPES) * len(ALL_SPLITS)
total_runs = len(SEEDS) * configs

print("=" * 70)
print(f"  SPARSE KAN v2 (RETRAIN-ONLY): {total_runs} runs")
print(f"  Reusing hyperparameters from: {ORIGINAL_RESULTS_DIR}")
print(f"  Writing NEW results to:       {RESULTS_DIR}")
print(f"  Original results directory is NEVER written to.")
print(f"  eps=1e-5 (was 0.1), init fix applied (was silently no-op)")
print("=" * 70)

all_results       = []
best_params_store = {}
completed         = 0
failed            = 0
total_start       = time.time()

for seed in SEEDS:
    set_seed(seed)
    seed_results_dir = RESULTS_DIR / f"seed_{seed}"

    print(f"\n\n{'═'*70}")
    print(f"  SEED {seed} — saving to {seed_results_dir}")
    print(f"{'═'*70}")

    for dataset in DATASETS:
        for target_type in TARGET_TYPES:
            for split_name in ALL_SPLITS:
                try:
                    exp = run_single_retrain(
                        split_name, dataset, target_type, device,
                        seed=seed,
                        seed_results_dir=seed_results_dir,
                    )

                    all_results.append({
                        "seed":    seed,
                        "dataset": dataset,
                        "split":   split_name,
                        "target":  target_type,
                        "used_l1": exp["used_l1"],
                        **{f"test_{k}": v for k, v in exp["metrics"]["test"].items()
                           if not isinstance(v, np.ndarray)},
                        "best_epoch": exp["best_epoch"],
                        "time_s":     exp["total_time"],
                    })

                    key = (seed, dataset, target_type, split_name)
                    best_params_store[key] = {**exp["best_params"], "used_l1": exp["used_l1"]}
                    completed += 1

                    elapsed = time.time() - total_start
                    rate = elapsed / completed
                    remaining_est = rate * (total_runs - completed)
                    print(f"\n  ✓ Completed {completed}/{total_runs}  "
                          f"({elapsed/60:.1f}min elapsed, ~{remaining_est/60:.1f}min remaining)")

                except Exception as e:
                    failed += 1
                    print(f"\n  ✗ FAILED ({failed}): seed={seed} "
                          f"{dataset}/{split_name}/{target_type}: {e}")
                    import traceback
                    traceback.print_exc()
                    continue

total_time = time.time() - total_start
print(f"\n\n{'='*70}")
print(f"  FINISHED: {completed}/{total_runs} completed, {failed} failed")
print(f"  Total time: {total_time/60:.1f} minutes")
print(f"{'='*70}")

# %%
if all_results:
    results_df = pd.DataFrame(all_results)
    raw_path = RESULTS_DIR / "all_seeds_raw.csv"
    results_df.to_csv(raw_path, index=False)
    print(f"Raw results saved to {raw_path}")

    binary_df = results_df[results_df["target"] == "binary"]
    print("\nBinary Test AUC (mean ± std across seeds):")
    if "test_auc" in binary_df.columns:
        agg = binary_df.groupby(["dataset", "split"])["test_auc"].agg(["mean", "std"])
        print(agg.to_string())

    cont_df = results_df[results_df["target"] == "continuous"]
    print("\nContinuous Test R² (mean ± std across seeds):")
    if "test_r2" in cont_df.columns:
        agg = cont_df.groupby(["dataset", "split"])["test_r2"].agg(["mean", "std"])
        print(agg.to_string())



# %% [markdown]
# ## Backtests (Seed-Averaged Signal)
#
# Same methodology as the original sweep: predictions averaged across the
# 3 seeds before backtesting. Reads from THIS notebook's v2 results
# directory only -- does not touch or read from the original sparse_kan/
# results in any way.
#
# NOTE: evaluation.py's fixed continuous threshold grids still assume a
# roughly [-5, +2] z-score range from the old target -- grid-edge warnings
# here reflect that KNOWN, DEFERRED issue, not a new bug.

# %%
def load_averaged_predictions(model_name, split_name, target_type, part,
                               seeds, results_dir):
    signals = []
    returns = None

    for seed in seeds:
        seed_dir = results_dir / f"seed_{seed}"
        loaded = load_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type=target_type,
            part=part,
            results_dir=seed_dir,
        )
        preds = loaded["predictions"]

        if returns is None:
            returns = preds["daily_return"].values

        if target_type == "binary":
            signals.append(preds["y_prob"].values)
        else:
            signals.append(preds["y_pred"].values)

    avg_signal = np.mean(np.stack(signals, axis=0), axis=0)
    return returns, avg_signal


def _make_json_safe(obj):
    if isinstance(obj, dict): return {k: _make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, pd.DataFrame): return obj.reset_index().to_dict(orient="records")
    if isinstance(obj, (np.floating, np.integer)): return obj.item()
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj


# %%
print("\n" + "=" * 70)
print("  SPARSE KAN v2 — BACKTESTS (seed-averaged signal)")
print("  Signal = mean prediction across seeds 42, 123, 456")
print("  Reading from v2 results only -- original sparse_kan/ untouched")
print("=" * 70)

backtest_dir = RESULTS_DIR / "backtests"
backtest_dir.mkdir(parents=True, exist_ok=True)

backtest_rows    = []
backtest_records = {}

for dataset in DATASETS:
    for split_name in ALL_SPLITS:
        model_name = f"sparse_kan_{dataset}"

        val_ret,  val_sig  = load_averaged_predictions(
            model_name, split_name, "binary", "val",  SEEDS, RESULTS_DIR)
        test_ret, test_sig = load_averaged_predictions(
            model_name, split_name, "binary", "test", SEEDS, RESULTS_DIR)

        bt_binary = run_full_backtest(
            val_returns=val_ret,   val_signal=val_sig,
            test_returns=test_ret, test_signal=test_sig,
            go_cash_when="above",
            model_name=f"{model_name}_v2 (binary)",
            split_name=split_name,
        )

        val_ret,  val_sig  = load_averaged_predictions(
            model_name, split_name, "continuous", "val",  SEEDS, RESULTS_DIR)
        test_ret, test_sig = load_averaged_predictions(
            model_name, split_name, "continuous", "test", SEEDS, RESULTS_DIR)

        bt_continuous = run_full_backtest(
            val_returns=val_ret,   val_signal=val_sig,
            test_returns=test_ret, test_signal=test_sig,
            go_cash_when="below",
            model_name=f"{model_name}_v2 (continuous)",
            split_name=split_name,
        )

        key = f"{dataset}/{split_name}"
        backtest_records[key] = {"binary": bt_binary, "continuous": bt_continuous}

        for target_type, bt in [("binary", bt_binary), ("continuous", bt_continuous)]:
            for strategy_name, strategy_key in [("simple", "simple"), ("risk_scaled", "risk_scaled")]:
                bt_result = bt[strategy_key]
                backtest_rows.append({
                    "dataset": dataset, "split": split_name,
                    "target_type": target_type, "strategy": strategy_name,
                    "sharpe": bt_result["sharpe"], "sortino": bt_result["sortino"],
                    "annual_return": bt_result["annual_return"],
                    "max_drawdown": bt_result["max_drawdown"],
                    "cumulative_return": bt_result["cumulative_return"],
                    "avg_exposure": bt_result["avg_exposure"],
                    "annual_turnover": bt_result["annual_turnover"],
                    "buy_hold_sharpe": bt_result["buy_hold_sharpe"],
                    "buy_hold_sortino": bt_result["buy_hold_sortino"],
                    "buy_hold_cumulative": bt_result["buy_hold_cumulative"],
                })

backtest_summary_df = pd.DataFrame(backtest_rows)
backtest_summary_path = backtest_dir / "backtest_summary.csv"
backtest_summary_df.to_csv(backtest_summary_path, index=False)
print(f"\n  Backtest summary saved to {backtest_summary_path}")

backtest_json_path = backtest_dir / "backtest_full_results.json"
with open(backtest_json_path, "w") as f:
    json.dump(_make_json_safe(backtest_records), f, indent=2, default=str)
print(f"  Full backtest results saved to {backtest_json_path}")


# %% [markdown]
# ## File Inventory

# %%
print("\n" + "=" * 70)
print("  SAVED FILES (v2 -- on Google Drive)")
print("=" * 70)

for seed in SEEDS:
    seed_dir = RESULTS_DIR / f"seed_{seed}"
    print(f"\n  ── seed_{seed}/ ──")
    for subdir in ["predictions", "metrics", "checkpoints"]:
        d = seed_dir / subdir
        if d.exists():
            files = list(d.glob("sparse_kan_*"))
            print(f"    {subdir}/: {len(files)} files")
        else:
            print(f"    {subdir}/: (not yet created)")

for fname in ["all_seeds_raw.csv"]:
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        print(f"\n  {fname}: ✓")
    else:
        print(f"\n  {fname}: (not yet created)")

print(f"\n  backtests/: {'✓' if (RESULTS_DIR / 'backtests' / 'backtest_summary.csv').exists() else '(not yet created)'}")

print(f"\n  Original results directory (unchanged, kept as ablation baseline):")
print(f"    {ORIGINAL_RESULTS_DIR}")


# %% [markdown]
# ## Disconnect Runtime

# %%
print("All experiments complete. Disconnecting runtime...")
from google.colab import runtime
runtime.unassign()

Mounted at /content/drive
PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
SANITY CHECK: confirming init fix + eps=1e-5 are both present
  ✓ bn1.eps = 1e-05, bn2.eps = 1e-05
  ✓ bn1.affine = False, bn2.affine = False
  ✓ init fix on base_weight: ratio = 0.976 (fan_in=5, expect ~0.83)
  ✓ init fix on spline_scaler: ratio = 0.989 (fan_in=5, expect ~0.83)
  ✓ All masked parameters are exactly zero and finite
  ✓ forward pass OK in train mode (batch=16)

SANITY CHECK PASSED -- safe to proceed with retrain-only sweep
Loading taxonomies...
  agg_full_moments: 1699 features, 331 subthemes, 13 themes
  agg_means: 574 features, 128 subthemes, 13 themes
Device: Tesla T4 (CUDA)
  SPARSE KAN v2 (RETRAIN-ONLY): 48 runs
  Reusing hyperparameters from: /content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan
  Writing NEW results to:       /content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan_v2
  Original results directory is NEVER written to.
  eps=1e-5 (was